# Yaobi 骨科智能体 · Colab 快速上手

<p align="center">
  <b>证据受控 · 医师在环 · 故障关闭</b><br>
  自主规划 · 许可门控的真实知识库 · 骨科用药安全 · 可视化控制台
</p>

这个 notebook 会把整套系统跑起来：

| 步骤 | 内容 |
| --- | --- |
| 1 | 安装与自检（128 个测试，无需网络） |
| 2 | 无 LLM 的确定性运行——红旗筛查、故障关闭、角色裁剪 |
| 3 | 骨科用药相互作用规则包（18 条规则 / 30 个药物类别） |
| 4 | 构建许可门控的知识库（openFDA / DailyMed / RxNorm 实时拉取） |
| 5 | 授权药典范围如何改变处方放行决策 |
| 6 | 接入 LLM（Azure / Poe / MiniMax / LiteLLM）做自主规划 |
| 7 | **在 Colab 内嵌启动可视化控制台** |

> ⚠️ **本项目不能用于真实临床决策或患者处方。** 指南与药典默认是占位数据；
> 内置规则包必须经本机构药师/医师复核后启用；所有含剂量输出都必须由医师逐味审核签名。

## 1 · 安装与自检

In [ ]:
#@title 安装（约 20 秒）
import os, sys, subprocess, pathlib

REPO = "https://github.com/psknlr/YaoBi-Harness.git"
BRANCH = "claude/orthopedic-agent-review-yupwfu"   #@param {type:"string"}
ROOT = pathlib.Path("/content/YaoBi-Harness")

if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# 稳定假名化密钥是硬性要求：缺失时加载病例库会直接报错，而不是静默使用随机密钥。
os.environ.setdefault("YAOBI_DEID_KEY", "colab-demo-key-change-me")
os.environ.setdefault("YAOBI_DEPLOYMENT_MODE", "research_noncommercial")

import yaobi_harness
print("yaobi-harness", yaobi_harness.__version__, "| cwd:", os.getcwd())

In [ ]:
#@title 跑一遍完整测试（128 个用例，不需要网络）
!python -m unittest discover -s tests 2>&1 | tail -5

## 2 · 确定性运行

未配置 LLM 时系统走**完全确定性**的规则路径。先看三个决定安全性的行为。

In [ ]:
#@title 红旗筛查：子句级否定，宁可多报不可漏报
from yaobi_harness.safety import red_flags

CASES = [
    "既往体健，现突发胸痛、大汗、呼吸困难",      # 历史前缀不得抑制当前急症
    "去年做过腰椎手术，今天突然不能排尿、会阴麻木",
    "多年前有腰痛史，现在双腿越来越无力",
    "腰痛，屁股和大腿根发麻，尿憋不住",           # 口语化表达
    "腰痛，无发热、无外伤、无大小便失禁、无会阴麻木",  # 真阴性
    "父亲患癌，本人只是久坐腰酸",                 # 家族史
    "如果以后胸痛怎么办，目前无不适",             # 假设语境
    "单纯夜间腰痛",                              # 弱信号 → 需线下检查，而不是丢弃
]
for text in CASES:
    r = red_flags.screen(text)
    tag = "🔴 URGENT " if r.urgent else ("🟡 SOFT   " if r.soft_hits else "🟢 ROUTINE")
    hits = ", ".join(sorted({h.signal for h in r.hits + r.soft_hits})) or "—"
    print(f"{tag} {hits:28s} {text}")

In [ ]:
#@title 端到端：急症患者 → 行动计划，且绝不出处方
import json
from yaobi_harness.graph import YaobiGraphRunner
from yaobi_harness.state import ClinicalRunState
from yaobi_harness.render import render

state = ClinicalRunState("突发腰痛伴尿潴留和会阴麻木", role="patient")
out = YaobiGraphRunner().run(state, allow_prescription=True)   # 即使显式允许开方

print("放行状态:", out.release_status)
print("风险模式:", out.risk_mode)
print("有处方草案:", "prescription_draft" in out.outputs)
print()
print(json.dumps(render(out, "patient")["urgent"], ensure_ascii=False, indent=2)[:900])

In [ ]:
#@title 角色裁剪：同一次运行，患者与医师看到的东西不同
from yaobi_harness.tools import ToolRegistry

RAW = {"病案号": "50512983", "姓名": "张三", "性别": "男", "年龄": "63岁",
       "主诉": "右腰部疼痛10天", "现病史": "久坐后疼痛明显。舌略暗。",
       "中医诊断": "腰痹/证型：气血痹阻证", "西医诊断": "腰痛",
       "中药": "1/独活*1克/10克/用法：无/贴数:7\n,2/盐杜仲*1克/12克/用法：无/贴数:7"}

tools = ToolRegistry(records=[RAW])
out = YaobiGraphRunner(tools).run(ClinicalRunState("腰痛，久坐后疼痛明显", role="patient"))

patient = render(out, "patient")
physician = render(out, "physician")
print("患者视图字段 :", sorted(patient))
print("医师视图字段 :", sorted(physician))
print()
print("患者视图是否包含他人病例/证据台账:",
      "research_patient_id" in json.dumps(patient, ensure_ascii=False), "/", "evidence_ledger" in patient)
print("医师视图证据条数:", len(physician["evidence_ledger"]))

## 3 · 骨科用药相互作用规则包

18 条规则、30 个药物类别，中英双语匹配。条件门控规则（双膦酸盐肾功能、
罗莫佐单抗心血管、椎管内麻醉等）只在提供对应患者状态时触发，避免误报。

In [ ]:
#@title 规则包速查
from yaobi_harness.knowledge import ortho_interactions as oi

print(json.dumps(oi.rule_pack_summary(), ensure_ascii=False, indent=2))
print()
CHECKS = [
    (["布洛芬", "华法林"], []),
    (["ibuprofen", "enalapril", "furosemide"], []),           # 三重打击
    (["羟考酮", "阿普唑仑"], []),                              # 呼吸抑制
    (["曲马多", "舍曲林"], []),                                # 血清素综合征
    (["阿仑膦酸钠", "碳酸钙"], []),                            # 吸收下降
    (["阿仑膦酸钠"], ["renal_impairment"]),                    # 条件门控
    (["秋水仙碱", "克拉霉素"], []),
    (["利伐沙班"], ["planned_neuraxial_anesthesia"]),
    (["对乙酰氨基酚", "茯苓"], []),                            # 应无发现
]
for meds, conds in CHECKS:
    hits = oi.evaluate(meds, conds)
    label = ", ".join(f"{h['rule_id']}/{h['severity']}" for h in hits) or "无发现"
    print(f"{str(meds) + (str(conds) if conds else ''):58s} → {label}")

In [ ]:
#@title 一次发现的完整内容：机制 + 处理 + 命中药物
finding = oi.evaluate(["羟考酮 5mg q12h", "阿普唑仑 0.4mg qn"])[0]
print(json.dumps(finding, ensure_ascii=False, indent=2))

In [ ]:
#@title 用药安全如何改变一次真实运行的放行状态
state = ClinicalRunState("腰痛3月，久坐加重", role="patient")
state.facts["medications"] = ["布洛芬 0.3g bid", "华法林 3mg qd"]
out = YaobiGraphRunner().run(state)

print("放行状态:", out.release_status)          # 由 needs_more_information 抬到 needs_examination
print("安全问题:", out.safety_issues)
print()
print("患者看到的通俗提醒:")
print(json.dumps(render(out, "patient")["medication_warnings"], ensure_ascii=False, indent=2))

## 4 · 构建许可门控的知识库

仓库**只包含代码与许可模型，不包含任何第三方受版权内容**。许可在**写入时**强制执行：

* 非商业来源（WHO、DDInter）在 `commercial` 模式下写入直接抛异常；
* 只读来源（AAOS、中华医学会、NMPA 文件）只存标题/版本/链接/摘录，全文被丢弃；
* 须授权来源（NICE、中国药典、DrugBank、BNF）在登记授权声明前完全禁用。

In [ ]:
#@title 来源目录：谁可用、为什么不可用
from yaobi_harness.knowledge.ingest import list_sources
from yaobi_harness.knowledge.licensing import LicensePolicy, DeploymentMode

for mode in (DeploymentMode.RESEARCH, DeploymentMode.COMMERCIAL):
    print(f"── {mode.value} " + "─" * 40)
    for row in list_sources(LicensePolicy(mode)):
        flag = "✅" if row["enabled"] else "🚫"
        print(f"  {flag} {row['source_id']:26s} {row['reuse']:17s} {'' if row['enabled'] else row['reason']}")
    print()

In [ ]:
#@title 实时构建（openFDA + DailyMed + RxNorm，均为公有领域/开放许可）
from yaobi_harness.knowledge.ingest import open_store, build

STORE = "/content/knowledge.db"
store = open_store(STORE)
report = build(
    store,
    ingredients=["ibuprofen", "warfarin sodium", "alendronate sodium",
                 "tramadol hydrochloride", "colchicine", "denosumab"],
    cache_dir="/content/.kcache",
)
print(json.dumps(report["built"], ensure_ascii=False, indent=2))
print("跳过:", [(s["source"], s["reason"]) for s in report["skipped"]])
print("统计:", report["stats"]["counts"])

In [ ]:
#@title 取回的真实说明书：带标签版本与检索时间
labels = store.label_sections("warfarin sodium", ["drug_interactions", "contraindications"])
for section in labels:
    print(f"── {section['section']}  (label v{section['label_version']}, {section['effective_time']})")
    print("  ", section["text"][:220], "…")
    print("   出处:", section["provenance"]["source"], "|", section["provenance"]["license"])
    print()

In [ ]:
#@title 许可拒绝是真的会抛异常，不是提示
from yaobi_harness.knowledge.store import KnowledgeStore
from yaobi_harness.knowledge.licensing import LicenseError, Attestation

commercial = KnowledgeStore(":memory:", LicensePolicy(DeploymentMode.COMMERCIAL))
for source in ("openfda", "ddinter", "who_guidelines", "chp_2025"):
    try:
        commercial.register_source(source)
        print(f"  ✅ {source:18s} 允许写入")
    except LicenseError as exc:
        print(f"  🚫 {source:18s} {exc}")

print()
print("只读来源：全文被丢弃，引用保留")
link_only = KnowledgeStore(":memory:", LicensePolicy(DeploymentMode.RESEARCH))
link_only.add_guideline("cma_guidelines", "CMA-LBP", "中国腰痛诊疗指南",
                        topic="腰痛", url="https://example.org/g",
                        body="这是受版权保护的全文，不应入库" * 10,
                        recommendations=["先排除红旗信号"])
hit = link_only.search_guidelines("腰痛")[0]
print("  has_full_text:", hit["has_full_text"], "| 摘录:", hit["recommendations"])

## 5 · 授权药典范围如何改变放行决策

这是整套系统里最关键的安全门槛：**拟用剂量必须落在授权范围内**，
而不只是"该药材有范围"。下面用同一批专家病例，只改药典范围，看结论如何翻转。

In [ ]:
#@title 同样的病例，30g vs 3–9g 范围
from yaobi_harness.knowledge.licensing import Attestation

HERBS = ["独活","桑寄生","杜仲","牛膝","当归","川芎","白芍","熟地黄","党参","茯苓","甘草","桃仁","红花","延胡索"]

def expert_cases(dose, n=6):
    body = lambda: "".join(f",{i}/{h}*1克/{dose}克/用法：无/贴数:7\n" for i, h in enumerate(HERBS, 1))
    return [{"病案号": f"C{i}", "年龄": "63岁", "主诉": "腰痛",
             "中医诊断": "腰痹/证型：气滞血瘀证", "中药": body()} for i in range(n)]

def physician_case():
    s = ClinicalRunState("腰痛3月，刺痛固定，久坐加重", role="physician")
    s.facts.update({"special_population": {"pregnancy": False, "age": 63,
                                           "renal": "normal", "liver": "normal"},
                    "medications_confirmed": True, "allergies_confirmed": True})
    return s

# 部署方持有《中国药典》授权后，登记授权声明
policy = LicensePolicy(DeploymentMode.RESEARCH,
                       {"chp_2025": Attestation("Colab 演示", "CHP-2025-DEMO", "2030-01-01")})

for label, low, high, dose in [("范围 3–9 g，专家用 30 g", 3.0, 9.0, 30.0),
                               ("范围 3–15 g，专家用 9 g", 3.0, 15.0, 9.0)]:
    pharm = KnowledgeStore(":memory:", policy)
    for herb in HERBS:
        pharm.add_dose_range("chp_2025", herb, low, high, basis="《中国药典》2025 一部", version="2025")
    out = YaobiGraphRunner(ToolRegistry(records=expert_cases(dose), knowledge=pharm)) \
            .run(physician_case(), allow_prescription=True)
    print(f"── {label}")
    print("   放行状态 :", out.release_status)
    print("   有草案   :", "prescription_draft" in out.outputs)
    if out.safety_issues:
        print("   阻断原因 :", out.safety_issues[0][:90], "…")
    print()

In [ ]:
#@title 通过门槛后的草案：每一味都带授权范围、样本量与证据 ID
pharm = KnowledgeStore(":memory:", policy)
for herb in HERBS:
    pharm.add_dose_range("chp_2025", herb, 3.0, 15.0, basis="《中国药典》2025 一部", version="2025")

out = YaobiGraphRunner(ToolRegistry(records=expert_cases(9.0), knowledge=pharm)) \
        .run(physician_case(), allow_prescription=True)
draft = out.outputs["prescription_draft"]

print("放行状态:", out.release_status, "| 不确定性:", draft["overall_uncertainty"])
print("指纹:", draft["prescription_hash"], "\n")
print(f"{'药味':<10}{'剂量':>8}{'授权范围':>14}{'样本量':>8}")
for herb in draft["herbs"][:6]:
    rng = "–".join(map(str, herb["authorized_range_g"])) + " g"
    print(f"{herb['herb_name']:<10}{herb['dose_value']:>6} g{rng:>14}{herb['sample_n']:>8}")

from yaobi_harness.render import citation_bundle
print("\n出处:")
for c in citation_bundle(out):
    print("  ", c.get("source"), "|", c.get("license"), "| 版本", c.get("version"))

In [ ]:
#@title 医师逐味审核签名 → approved_by_physician
approving = physician_case()
approving.facts["physician_review"] = {
    "physician_id": "D-10086",
    "signature": "demo-signature",
    "approvals": {h["herb_name"]: True for h in draft["herbs"]},
}
approved = YaobiGraphRunner(ToolRegistry(records=expert_cases(9.0), knowledge=pharm)) \
             .run(approving, allow_prescription=True)
print("放行状态:", approved.release_status)
print("审核结果:", approved.outputs["physician_review"])

## 6 · 接入 LLM

支持 **Azure OpenAI / Poe / MiniMax / LiteLLM**，纯标准库 HTTP，无额外依赖。

LLM 在本系统中是**只能加安全、不能减安全**的顾问：它可以提议计划、追加红旗、
追加安全异议，但不能发明 Agent、不能触及技能未授权的工具、不能清除规则层命中的
风险信号、不能生成剂量。任何越权提案会被**整体驳回**并回退确定性计划。

In [ ]:
#@title 配置 provider（留空则以确定性规则路径运行）
PROVIDER = "none"  #@param ["none", "azure", "poe", "minimax", "litellm"]
API_KEY  = ""      #@param {type:"string"}
MODEL    = ""      #@param {type:"string"}
BASE_URL = ""      #@param {type:"string"}
EXTRA    = ""      #@param {type:"string"}

import os
os.environ["YAOBI_LLM_PROVIDER"] = PROVIDER
if PROVIDER == "azure":
    os.environ["AZURE_OPENAI_API_KEY"] = API_KEY
    os.environ["AZURE_OPENAI_ENDPOINT"] = BASE_URL      # https://xxx.openai.azure.com
    os.environ["AZURE_OPENAI_DEPLOYMENT"] = MODEL       # 部署名
elif PROVIDER == "poe":
    os.environ["POE_API_KEY"] = API_KEY
    os.environ["POE_MODEL"] = MODEL or "Claude-Sonnet-4.5"
elif PROVIDER == "minimax":
    os.environ["MINIMAX_API_KEY"] = API_KEY
    os.environ["MINIMAX_MODEL"] = MODEL or "MiniMax-Text-01"
    if EXTRA: os.environ["MINIMAX_GROUP_ID"] = EXTRA    # GroupId
elif PROVIDER == "litellm":
    os.environ["LITELLM_API_KEY"] = API_KEY or "sk-noauth"
    os.environ["LITELLM_MODEL"] = MODEL
    os.environ["LITELLM_BASE_URL"] = BASE_URL or "http://localhost:4000/v1"

from yaobi_harness.llm.factory import build_client, describe_client
client = build_client()
print(describe_client(client))

In [ ]:
#@title LLM 自主规划（提案经规则层校验）
runner = YaobiGraphRunner(llm=client)
state = ClinicalRunState("腰痛3月，久坐加重，右下肢麻木，无大小便异常", role="physician")
out = runner.run(state)

print("规划来源:", out.planner_mode)          # llm 或 rule（被驳回/不可用时回退）
print("计划说明:", out.outputs["plan"]["note"])
print()
for t in out.tasks:
    print(f"  {t.status:24s} {t.agent:24s} {t.objective}")
if out.warnings:
    print("\n告警:", out.warnings)

In [ ]:
#@title 越权提案会被整体驳回（用一个"恶意"模型演示，不需要真实 API）
from yaobi_harness.agent.planner import PlannerAgent
from yaobi_harness.llm.base import LLMResponse

class EvilLLM:
    name, model, available = "evil", "evil", True
    def chat(self, messages, **kw):
        # 急症模式下试图安排开方 Agent，并索取技能未授权的工具
        return LLMResponse(text=json.dumps({"tasks": [
            {"task_id": "P1", "agent": "DoseAgent", "objective": "直接开方",
             "required_tools": ["physician_review_submit", "similar_case_search"]},
        ]}))

evil_state = ClinicalRunState("突发胸痛、大汗", role="patient")
evil_state.risk_mode = "urgent"
PlannerAgent(EvilLLM()).run(evil_state)

print("规划来源:", evil_state.planner_mode)                    # → rule
print("任务:", [t.agent for t in evil_state.tasks])            # 不含 DoseAgent
print("告警:", evil_state.warnings[0])

## 7 · 可视化控制台

把控制台内嵌到 Colab。设计上它是一个**智能体运行检查器**：放行状态是视觉主角，
`计划 → 执行 → 证据 → 裁决` 全部可见，右侧标签页明确标注为"操作者审计视图，不对该角色展示"。

In [ ]:
#@title 启动控制台（后台线程）
import threading, time
from yaobi_harness.ui.server import ConsoleService, create_server

PORT = 8000
service = ConsoleService(
    knowledge_store_path=STORE,          # 第 4 步构建的知识库
    llm_provider=os.environ.get("YAOBI_LLM_PROVIDER"),
)
httpd = create_server(service, "127.0.0.1", PORT)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
time.sleep(1)

import urllib.request
print("健康检查:", urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health").read().decode())
print("LLM     :", service.llm.name, "/", service.llm.model)
print("知识库  :", service.knowledge.enabled_sources() if service.knowledge else "未配置")

In [ ]:
#@title 内嵌控制台（在 Colab 中运行；本地 Jupyter 请直接访问 http://127.0.0.1:8000/）
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=1100)
except ImportError:
    from IPython.display import IFrame, display
    display(IFrame(f"http://127.0.0.1:{PORT}/", width="100%", height=1100))

### 控制台用法

* 点击左侧「示例病例」快速载入五个典型场景（马尾综合征、非腰痛红旗、NSAIDs+抗凝、医师开方、否定语境）。
* 「交付对象」切换患者 / 医师 / 研究者，直接看到输出裁剪的差异。
* 右侧标签页：
  * **交付内容** — 该角色实际会收到的东西
  * **规划与执行** — 任务图、每个 Agent 的状态与工具调用
  * **用药安全** — 相互作用发现，含机制、处理与命中药物
  * **证据台账** — 每条证据的等级与是否可放行，以及结论↔证据的绑定
  * **安全审查** — 终结节点的裁决、修复请求与已执行检查
* 顶部「用药速查」不跑完整病例，只做相互作用筛查；「知识库」展示来源目录、许可状态与规则包全文。

## 也可以直接用命令行

```bash
export YAOBI_DEID_KEY="$(openssl rand -hex 32)"

python -m yaobi_harness run --role physician --complaint "腰痛3月，久坐加重" \
    --knowledge-store ./knowledge.db --allow-prescription
python -m yaobi_harness knowledge sources
python -m yaobi_harness knowledge check-interactions --medications 布洛芬 华法林
python -m yaobi_harness ui --port 8000 --knowledge-store ./knowledge.db
```

## 还没做完的部分

LangGraph 原生 interrupt/resume、医师审批 UI、多轮问诊状态机、中文指南的结构化推荐抽取、
大规模对抗性安全评测与红旗召回率基线。**本项目不能对外宣称为临床可用系统。**